In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib agg
    
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
       
from HPIB.HP4155 import HP4155
from HPIB.HPT import Plot, PlotVgs, PlotVp, CalcIsSat, SecDer, Plot2P, PlotDiode4P
from HPIB.DevParams import UMC
from HPIB.INOSerial import Arduino

from BFmodule import DR

from YFunc import YFuncExtraction

from Test import TestDevice, WriteLog

from IPython.display import clear_output, display
from os import makedirs, rename
from time import sleep
from datetime import datetime, timedelta

Elsa=DR()

HP=HP4155("GPIB0::17", debug=False)
HP.IntTime="LONG"
HP.reset()

INO=Arduino("COM3")
print(INO.ask('*'.encode()))

Elsa v2.4.2
HEWLETT-PACKARD,4155A,0,01.04:01.04:01.00
InoMatrix



In [2]:
def WriteLog(msg, path, mode='a', end='\n', output=True):
    with open(path, mode) as logfile:
        logfile.write(msg+end)
    if output:
        print(f"{msg}{end}", end='')

def TestDevice(device, chn, path, params, HiPot=False):
    global HP, INO, Elsa, prog_bar
    INO.opench(chn+1)

    if not device:
        INO.opench(0)
        sleep(2)
        return 0

    if device[:2].upper() not in ['CA', 'CB', 'CG', 'TP', 'TN', 'DP', 'DN']:
        return "Invalid device"

    WriteLog(f"## Ch {chn+1} {device}", path + 'log.txt')
    pathp=path+device
    makedirs(pathp, exist_ok=True)
    
    ####################### Measure Diode
    #
    #
    if 'D' in device.upper():
        
        # HP.StopCond="COMP"
        HP.IntTime="MED"
        HP.Diode4P(params['Vfmin'], params['Vfmax'], params['Vfstep'], Comp=params['IComp'] if not HiPot else 2*params['IComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M%S')
        V100, V10=PlotDiode4P(HP.SingleSave(f"{pathp}/{now}.csv", timeout=30, real=True))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/{now}.csv", f"{pathp}/Diode - {temp} - {now}.csv")
            rename(f"{pathp}/{now}.png", f"{pathp}/Diode - {temp} - {now}.png")
            rename(f"{pathp}/{now} log.png", f"{pathp}/Diode - {temp} - {now} log.png")
            
            with open(f"{pathp}/V10010uA.log", 'a') as DiodeParam:
                DiodeParam.write(f"{temp},{format(V100, '.3f')},{format(V10, '.3f')}\n")
        except Exception as err:
            print (">>> Error:", err)
        
        now=datetime.now()
            
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        HP.StopCond="OFF"
        HP.IntTime="LONG"
        
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################

    ####################### Measure Transistor
    #
    #

    if 'T' in device.upper():
        
        ptype='P' in device.upper()
        
        HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vd'], ptype=ptype)
  
        now=datetime.now().strftime('%y%m%d %H%M')
        LIN = PlotVgs(HP.SingleSave(f"{pathp}/IdVgs - {now}.csv", timeout=30))
        
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            newname=f"{pathp}/IdVgs - {temp} - {now}"
            rename(f"{pathp}/IdVgs - {now}.csv", f"{newname}.csv")
            rename(f"{pathp}/IdVgs - {now}.png", f"{newname}.png")
        except Exception as err:
            print (">>> Error:", err)
        now=datetime.now()

        try:
            LIN, Vth, SS, migm, miyf, theta1, theta2, errmax = YFuncExtraction(f"{newname}.csv", UMC[int(device[2:])], 4.2, 3.9, params['Vd'])
            WriteLog(f"LIN={format(LIN, '.3f')}, Vth={format(Vth, '.3f')} V, SS={format(SS, '.2f')} mV/dec, miyf={format(migm, '.1f')}, miyf={format(miyf, '.1f')}, theta1={format(theta1, '.3e')}, theta2={format(theta2, '.3e')}", path + 'log.txt')
            WriteLog(f"{temp},{format(LIN, '.3f')},{format(Vth, '.3f')},{format(SS, '.2f')},{format(migm, '.1f')},{format(miyf, '.1f')},{format(theta1, '.3e')},{format(theta2, '.3e')}", pathp + '/params.txt')
        except Exception as err:
            print (">>> Error:", err)
            WriteLog(f"Vth={LIN} V", path + 'log.txt')
        
        while (datetime.now()-now).seconds < params['min_wait']:
            prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
            sleep(0.5)
        prog_bar.update("Measuring")
            
        if HiPot:
            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVgs(params['Vmin'], params['Vmax'], params['Vstep'], params['Vmax'], ptype=ptype, sat=True)
            HP.SingleSave(f"{pathp}/IdVgsSat - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVgsSat - {now}.csv", f"{pathp}/IdVgsSat - {temp} - {now}.csv")
            except Exception as err:
                print(">>> Error:", err)
            n, Ispec = CalcIsSat(f"{pathp}/IdVgsSat - {temp} - {now}.csv",temp)
            WriteLog(f"n={format(n, '.3f')}, Ispec={format(Ispec, '.3e')} A", path + 'log.txt')

            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
            if Ispec != 0:
                now=datetime.now().strftime('%y%m%d %H%M')
                HP.SetVp(Ispec, params['Vmin'], params['Vmax'], 0.05, ptype=ptype)
                HP.SingleSave(f"{pathp}/VpVg - {now}.csv", timeout=30)
                try:
                    temp=format(Elsa.GetT('t4k'), '07.3f')
                    rename(f"{pathp}/VpVg - {now}.csv", f"{pathp}/VpVg - {temp} - {now}.csv")
                except Exception as err:
                    print (">>> Error:", err)
                VTO=PlotVp(f"{pathp}/VpVg - {temp} - {now}.csv")
                WriteLog(f"VTO={VTO} V", path + 'log.txt')

                now=datetime.now()
                while (datetime.now()-now).seconds < 4*params['min_wait']:
                    prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                    sleep(0.5)
                prog_bar.update("Measuring")

            now=datetime.now().strftime('%y%m%d %H%M')
            HP.SetVds(params['Vmin'], params['Vmax'], params['Vstep'], params['Vgmin'], params['Vgmax'], params['Vgstep'], ptype=ptype)
            HP.SingleSave(f"{pathp}/IdVds - {now}.csv", timeout=30)
            try:
                temp=format(Elsa.GetT('t4k'), '07.3f')
                rename(f"{pathp}/IdVds - {now}.csv", f"{pathp}/IdVds - {temp} - {now}.csv")
            except Exception as err:
                print (">>> Error:", err)
            Plot(f"{pathp}/IdVds - {temp} - {now}.csv", 'Vd', 'Id')
            
            now=datetime.now()
            while (datetime.now()-now).seconds < 4*params['min_wait']:
                prog_bar.update(f"Waiting: {4*params['min_wait']-(datetime.now()-now).seconds} s")
                sleep(0.5)
            prog_bar.update("Measuring")
            
        INO.opench(0)
        WriteLog('', path + 'log.txt')
        return 0
    #
    #
    #######################
        
    ####################### Measure CrossBridge
    #
    #
    if 'CG' in device.upper():
           
        now=datetime.now().strftime('%y%m%d %H%M')
        V10u=HP.MeasV10u(f"{pathp}/2P - {now}.csv", timeout=0.5)
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"V_10u={format(V10u, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/VxT 10uA.log", 'a') as VxT:
            VxT.write(f"{temp},{format(V10u, '.4e')}\n")
        
    if 'CB' in device.upper():
    
        HP.Set4P(params['Ifmin'], params['Ifmax'], params['Ifpoints'])
        
        now=datetime.now().strftime('%y%m%d %H%M%S')
        RCB=Plot2P(HP.SingleSave(f"{pathp}/4P - {now}.csv", timeout=30))
        RCB=RCB*1e3
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/4P - {now}.csv", f"{pathp}/4P - {temp} - {now}.csv")
            rename(f"{pathp}/4P - {now}.png", f"{pathp}/4P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"RCB={format(RCB, '7.3f')}", path + 'log.txt')
        with open(f"{pathp}/RxT 4P.log", 'a') as RxT4p:
            RxT4p.write(f"{temp},{format(RCB, '.2e')}\n")
            
    if 'C' in device.upper():

        HP.Set2P(params['Ifmin']/params['Iffactor'], params['Ifmax']/params['Iffactor'], params['Ifpoints'], SMUN='SMU1', SMUP='SMU2', Comp=params['VComp'])
            
        now=datetime.now().strftime('%y%m%d %H%M%S')
        Rshort=Plot2P(HP.SingleSave(f"{pathp}/2P - {now}.csv", timeout=30))
        try:
            temp=format(Elsa.GetT('t4k'), '07.3f')
            rename(f"{pathp}/2P - {now}.csv", f"{pathp}/2P - {temp} - {now}.csv")
            rename(f"{pathp}/2P - {now}.png", f"{pathp}/2P - {temp} - {now}.png")
        except Exception as err:
            print (">>> Error:", err)
        WriteLog(f"Rshort={format(Rshort, '.2f')}", path + 'log.txt')
        with open(f"{pathp}/RxT 2P.log", 'a') as RxT2p:
            RxT2p.write(f"{temp},{format(Rshort, '.2f')}\n")

    now=datetime.now()
    while (datetime.now()-now).seconds < params['min_wait']:
        prog_bar.update(f"Waiting: {params['min_wait']-(datetime.now()-now).seconds} s")
        sleep(0.5)
    prog_bar.update("Measuring")
            
    INO.opench(0)
    WriteLog('', path + 'log.txt')
    return 0
    #
    #
    #######################

In [3]:
params = {
'Vd' : 0.05,
'Vmin' : 0,
'Vmax' : 1.5,
'Vstep' : 0.02,

'Vgmin' : 0.6,
'Vgmax' : 1.4,
'Vgstep' : 0.2,

'Vfmin' : 0,
'Vfmax' : 1.5,
'Vfstep' : 0.02,
'IComp' : 1e-3,

'Ifmin' : -5e-3,
'Ifmax' : 5e-3,
'Ifpoints' : 10,
'Iffactor' : 50,
'VComp' : 1.5,

'min_wait' : 15
}

DeviceList = ['CB1']
# DeviceList = ['', 'DN1']
prepath = "C:/Users/Zucchi/Documents/Medidas/250205 CB1/"

In [6]:
###### Warmup measurement

MsrNo=1

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150:
    path = prepath+"Cooldown/"
else:
    path = prepath+"Warmup/"
makedirs(path, exist_ok=True)

path = prepath+"Cooldown/"

HP.IntTime='LONG'
HP.LongNPLC=2

clear_output()

prog_bar=display('',display_id=True)

try:
    current_temp=Elsa.GetT('t4k')
except:
    pass

if current_temp > 150: finish_temp=5
else: finish_temp=295

finish_temp=3.5

last_temp=150

while (last_temp != finish_temp):
    #loop until temp changes and is a multiple of freq_temp
    start=datetime.now()
    while (np.around(current_temp, 1)==last_temp):
        if (datetime.now()-start).seconds/60 > 5: ## Failsafe trigger
            break
        for i in range(15):
            prog_bar.update(f"{datetime.now().strftime('%H:%M:%S')} - T = {format(current_temp, '.1f')} K")
            sleep(1)
    
        current_temp=Elsa.GetT('t4k')

    last_temp=np.around(current_temp, 1)
    
    for Msr in range(MsrNo):
        plt.close('all')
        clear_output()
        prog_bar=display('Measuring',display_id=True)

        start=datetime.now()
        WriteLog(f"# {current_temp} K - Measurement {Msr+1} - {datetime.now().strftime('%y%m%d %H%M')}\n", path+'log.txt')
        for chn, device in enumerate(DeviceList):
            if device:
                TestDevice(device, chn, path, params)
    
        WriteLog(f"{current_temp} K - Measurement {Msr+1} end . Duration: {int((datetime.now()-start).seconds/60)}:{(datetime.now()-start).seconds%60:02d}\n\n#######################\n", path + 'log.txt')

    try:
        current_temp=Elsa.GetT('t4k')
    except:
        pass
        if i < MsrNo:
            start=datetime.now()
            while (datetime.now()-start).seconds < MsrWait*60:
                prog_bar.update(f"Waiting: {int(MsrWait*60)-(datetime.now()-start).seconds} s")
                sleep(1)
prog_bar.update("Ramp measurement done")

'Ramp measurement done'

# 3.5389 K - Measurement 1 - 250208 0901

## Ch 1 CB1
Set 4P
I=(-0.005, 0.005), 10 Points
Done 4P. Duration: 3 s                                     
RCB=841.915
Set 2P - 12
I=(-0.0001, 0.0001), 10 Points
Done 2P - 12. Duration: 1 s                                     
Rshort=39.75

3.5389 K - Measurement 1 end . Duration: 0:29

#######################



In [35]:
HP.close()
INO.close()
print("Comm closed")

Comm closed
